# 🧠 Transfer Learning na prática — CIF-10 com ResNet-18
### Introdução à Ciência de Dados (ENG4502) — PUC-Rio

Neste notebook você vai usar uma rede **já treinada** (ResNet-18, treinada no ImageNet com 1,2 milhão de imagens) para classificar as 10 categorias do CIFAR-10 — **sem treinar a rede do zero**.

A estratégia se chama **Feature Extraction (extração de características)**:

1. **Congelamos** toda a rede pré-treinada e a usamos só como um "extrator de características".
2. Passamos as imagens pela rede **uma única vez** e guardamos os vetores de 512 números que ela produz (o *truque de cache* que deixa tudo rápido).
3. Treinamos apenas um **classificador linear simples** sobre esses vetores — isso leva segundos.

> ⏱️ Rodando na **GPU** do Colab, o notebook inteiro leva **menos de 1 minuto**.


## ▶️ Passo 0 — Ativar a GPU (importante!)

No menu: **Ambiente de execução → Alterar o tipo de ambiente de execução → Acelerador de hardware: GPU (T4)**.

Depois execute a célula abaixo para confirmar.

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'\u2705 GPU ativa: {torch.cuda.get_device_name(0)}')
else:
    device = torch.device('cpu')
    print('\u26a0\ufe0f  GPU N\u00c3O detectada.')
    print('   Menu: Ambiente de execu\u00e7\u00e3o \u2192 Alterar o tipo \u2192 GPU (T4)')
    print('   O notebook ainda funciona na CPU (gra\u00e7as ao cache de features), s\u00f3 um pouco mais lento.')

## 📦 Passo 1 — Importar as bibliotecas
Todas já vêm pré-instaladas no Colab — nada para instalar.

In [ ]:
import time, random
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset, TensorDataset
import matplotlib.pyplot as plt

torch.manual_seed(42)          # reprodutibilidade: todos veem o mesmo resultado
random.seed(42)

## 🖼️ Passo 2 — Baixar e preparar o CIFAR-10

O CIFAR-10 tem 10 classes de imagens 32×32. Precisamos de duas transformações:

- **`Resize(224)`** — a ResNet espera imagens 224×224 (tamanho do ImageNet).
- **`Normalize(...)`** — usamos a **média e desvio do ImageNet**. Isso é essencial: a rede foi treinada com imagens normalizadas assim, então precisamos apresentar nossas imagens da mesma forma.

In [ ]:
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_full = datasets.CIFAR10(root='data', train=True,  download=True, transform=transform)
val_full   = datasets.CIFAR10(root='data', train=False, download=True, transform=transform)

classes = ['avi\u00e3o','autom\u00f3vel','p\u00e1ssaro','gato','veado',
           'cachorro','sapo','cavalo','navio','caminh\u00e3o']
print('Classes:', classes)

### Subconjunto balanceado (para a aula ser rápida)

Usamos **500 imagens por classe** (5.000 no total) em vez das 50.000. Lemos os rótulos direto de `dataset.targets`, sem carregar as imagens — bem mais rápido.

> 💡 Quer treinar com o dataset completo? Troque `N_TRAIN` e `N_VAL` por `None`.

In [ ]:
def balanced_subset(dataset, n_total):
    if n_total is None:
        return dataset
    n_per_class = n_total // 10
    indices, counts = [], {c: 0 for c in range(10)}
    for idx, label in enumerate(dataset.targets):   # s\u00f3 r\u00f3tulos, sem abrir imagens
        if counts[label] < n_per_class:
            indices.append(idx)
            counts[label] += 1
        if len(indices) == n_total:
            break
    return Subset(dataset, indices)

N_TRAIN, N_VAL = 5000, 1000
train_dataset = balanced_subset(train_full, N_TRAIN)
val_dataset   = balanced_subset(val_full,   N_VAL)
print(f'Treino: {len(train_dataset)} imagens | Valida\u00e7\u00e3o: {len(val_dataset)} imagens')

## 🏗️ Passo 3 — Carregar a ResNet-18 pré-treinada e congelá-la

Removemos a última camada (`fc`, que classificava as 1000 classes do ImageNet) e ficamos com o **backbone**, que transforma cada imagem em um vetor de **512 características**. Congelamos tudo (`requires_grad = False`): esses pesos **não serão treinados**.

In [ ]:
weights = models.ResNet18_Weights.IMAGENET1K_V1
resnet  = models.resnet18(weights=weights)

# Backbone = ResNet sem a \u00faltima camada -> sa\u00edda de 512 dimens\u00f5es
backbone = nn.Sequential(*list(resnet.children())[:-1])
backbone.eval().to(device)
for p in backbone.parameters():
    p.requires_grad = False

print('Backbone pronto. Cada imagem vira um vetor de 512 n\u00fameros.')

## ⚡ Passo 4 — O truque de velocidade: extrair as características UMA vez

Como o backbone está **congelado**, a saída dele para cada imagem **nunca muda**. Então não faz sentido recalcular a cada época! Passamos todas as imagens pela rede **uma única vez** e guardamos os vetores de 512 dimensões na memória.

É isso que transforma minutos em **segundos**.

In [ ]:
@torch.no_grad()
def extract_features(dataset):
    loader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=0)  # 0 = compatível com Windows e Colab
    feats, labels = [], []
    for x, y in loader:
        f = backbone(x.to(device)).flatten(1)   # (lote, 512)
        feats.append(f.cpu())
        labels.append(y)
    return torch.cat(feats), torch.cat(labels)

t0 = time.time()
Xtr, ytr   = extract_features(train_dataset)
Xval, yval = extract_features(val_dataset)
print(f'Caracter\u00edsticas extra\u00eddas em {time.time()-t0:.1f}s')
print(f'Treino: {tuple(Xtr.shape)} | Valida\u00e7\u00e3o: {tuple(Xval.shape)}')

## 🎯 Passo 5 — Treinar o classificador linear

Agora treinamos só uma camada `Linear(512 → 10)` sobre os vetores já prontos. Como não há mais imagens passando pela rede pesada, cada época leva **frações de segundo** — por isso podemos rodar 15 épocas tranquilamente.

In [ ]:
clf       = nn.Linear(512, 10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(clf.parameters(), lr=0.01, momentum=0.9)

train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)
Xval_d, yval_d = Xval.to(device), yval.to(device)

EPOCHS = 15
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    clf.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(clf(xb), yb)
        loss.backward()
        optimizer.step()
    clf.eval()
    with torch.no_grad():
        acc = (clf(Xval_d).argmax(1) == yval_d).float().mean().item()
    print(f'\u00c9poca {epoch:2d}/{EPOCHS} | loss {loss.item():.4f} | val acc {acc*100:.1f}%')

print(f'\n\u2705 Treino conclu\u00eddo em {time.time()-t0:.1f}s | Acur\u00e1cia final: {acc*100:.1f}%')

> 💡 **Pare e reflita:** A rede acabou de atingir essa acurácia treinando apenas
> **5.130 parâmetros** em vez dos 11 milhões da ResNet completa.
>
> O que isso revela sobre o conhecimento já embutido nos pesos pré-treinados do ImageNet?

## 👁️ Passo 6 — Ver o modelo em ação

Vamos olhar 8 imagens de validação com a previsão do modelo. **Verde** = acertou, **vermelho** = errou.

In [ ]:
clf.eval()
idxs = random.sample(range(len(val_dataset)), 8)

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, i in zip(axes.flat, idxs):
    img, label = val_dataset[i]
    with torch.no_grad():
        feat = backbone(img.unsqueeze(0).to(device)).flatten(1)
        pred = clf(feat).argmax(1).item()
    show = (img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(show)
    ax.axis('off')
    cor = 'green' if pred == label else 'red'
    ax.set_title(f'prev: {classes[pred]}\nreal: {classes[label]}', color=cor, fontsize=11)
plt.tight_layout()
plt.show()

## 🏁 Conclusão

Com **Transfer Learning** chegamos a **~75% de acurácia** (15 épocas, menos de 1 minuto na GPU), treinando apenas **5.130 parâmetros** (a camada linear) em vez dos 11 milhões da rede inteira. Treinar do zero com tão poucos dados daria ~38%.

**A ideia-chave:** uma rede treinada em milhões de imagens já "sabe" reconhecer bordas, texturas e formas. Reaproveitamos esse conhecimento e só ensinamos a parte final.

---
### 🚀 Desafio (opcional)

1. Troque `N_TRAIN, N_VAL = 5000, 1000` por `None, None` e veja a acurácia subir (vai demorar mais).
2. Aumente `EPOCHS` e observe quando a acurácia para de melhorar.
3. **Fine-tuning:** em vez de congelar tudo, descongele as últimas camadas e treine a rede inteira com `lr` baixo (ex.: `1e-3`). Compare a acurácia.